# CFU Classifier — Gradio App
Run all cells top to bottom. A browser window will open when the last cell runs.

**First time only:** run the cell below to install dependencies.

In [ ]:
# First-time setup — run once, then skip
# %pip install ultralytics sahi ensemble-boxes gradio pandas

In [ ]:
# Uncomment if running on the cluster (sets CUDA paths)
# import os
# os.environ["CUDA_HOME"] = "/cm/shared/apps/cuda11.8/toolkit/11.8.0"
# os.environ["PATH"] = "/cm/shared/apps/cuda11.8/toolkit/11.8.0/bin:" + os.environ.get("PATH", "")
# os.environ["LD_LIBRARY_PATH"] = "/cm/shared/apps/cuda11.8/toolkit/11.8.0/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

In [10]:
from ultralytics import YOLO
from sahi.slicing import slice_image
from ensemble_boxes import weighted_boxes_fusion
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
import torchvision.ops as ops
import gradio as gr

In [11]:
# -- Configuration --
MODEL_PATH    = "runs/detect/3200px_training_v3/cfu_detector/weights/best.pt"
CONF          = 0.25
IOU           = 0.1
GRID          = 2
OVERLAP_RATIO = 0.2
METHOD        = 'wbf'

CLASS_NAMES = ['BFU', 'GM', 'E', 'GEMM']
COLORS      = ['#e6194b', '#3cb44b', '#4363d8', '#f58231']

In [12]:
model = YOLO(MODEL_PATH)
# model.to('cuda:0')  # Uncomment on cluster with GPU
print(f"Loaded: {MODEL_PATH}")
print(f"Device: {next(model.model.parameters()).device}")

Loaded: runs/detect/3200px_training_v3/cfu_detector/weights/best.pt
Device: cpu


In [13]:
def predict_full_image(image_path, conf=CONF, iou=IOU, grid=GRID,
                       overlap_ratio=OVERLAP_RATIO, method=METHOD):
    img = Image.open(image_path)
    W, H = img.size
    tile_w = W // grid
    tile_h = H // grid

    slice_result = slice_image(
        image=str(image_path),
        slice_height=tile_h,
        slice_width=tile_w,
        overlap_height_ratio=overlap_ratio,
        overlap_width_ratio=overlap_ratio,
        verbose=False,
    )

    all_boxes, all_scores, all_labels = [], [], []

    for sliced in slice_result.sliced_image_list:
        tx1 = sliced.starting_pixel[0]
        ty1 = sliced.starting_pixel[1]
        tile_img = Image.fromarray(sliced.image)
        results = model.predict(tile_img, conf=conf, iou=iou, verbose=False)
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            all_boxes.append([x1 + tx1, y1 + ty1, x2 + tx1, y2 + ty1])
            all_scores.append(float(box.conf[0]))
            all_labels.append(int(box.cls[0]))

    if not all_boxes:
        return np.empty((0, 4)), np.array([]), np.array([])

    if method == 'wbf':
        boxes_norm = (np.array(all_boxes) / np.array([W, H, W, H])).clip(0, 1).tolist()
        boxes_out, scores_out, labels_out = weighted_boxes_fusion(
            [boxes_norm], [all_scores], [all_labels],
            iou_thr=iou, skip_box_thr=conf,
        )
        boxes_out = np.array(boxes_out) * np.array([W, H, W, H])
        return boxes_out, np.array(scores_out), np.array(labels_out, dtype=int)
    else:
        boxes_t  = torch.tensor(all_boxes,  dtype=torch.float32)
        scores_t = torch.tensor(all_scores, dtype=torch.float32)
        labels_t = torch.tensor(all_labels, dtype=torch.int32)
        keep_indices = []
        for cls_idx in labels_t.unique():
            mask = labels_t == cls_idx
            keep = ops.nms(boxes_t[mask], scores_t[mask], iou)
            keep_indices.append(mask.nonzero(as_tuple=True)[0][keep])
        keep = torch.cat(keep_indices)
        return boxes_t[keep].numpy(), scores_t[keep].numpy(), labels_t[keep].numpy()

In [14]:
def render_boxes(image_path, df):
    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(np.array(Image.open(image_path)))
    for _, row in df.iterrows():
        cls = row['class']
        if cls not in CLASS_NAMES:
            continue
        color = COLORS[CLASS_NAMES.index(cls)]
        x1, y1, x2, y2 = int(row['x1']), int(row['y1']), int(row['x2']), int(row['y2'])
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1, edgecolor=color, facecolor='none'
        ))
        ax.text(x1, y1 - 4, cls, color=color, fontsize=6, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    return fig

def counts_from_df(df):
    counts = {name: 0 for name in CLASS_NAMES}
    for cls in df['class']:
        if cls in counts:
            counts[cls] += 1
    counts['total'] = sum(v for k, v in counts.items() if k != 'total')
    return counts

def run_inference(image_path):
    if image_path is None:
        return None, pd.DataFrame(columns=['class', 'x1', 'y1', 'x2', 'y2']), {}
    boxes, scores, labels = predict_full_image(image_path)
    df = pd.DataFrame(
        [[CLASS_NAMES[cls], int(x1), int(y1), int(x2), int(y2)]
         for (x1, y1, x2, y2), cls in zip(boxes, labels)],
        columns=['class', 'x1', 'y1', 'x2', 'y2']
    )
    return render_boxes(image_path, df), df, counts_from_df(df)

def update_preview(image_path, df):
    if image_path is None or df is None or len(df) == 0:
        return None, {}
    df = df.copy()
    df['class'] = df['class'].fillna('GM')
    return render_boxes(image_path, df), counts_from_df(df)

def save_annotations(image_path, df):
    if image_path is None or df is None or len(df) == 0:
        return "Nothing to save."
    img = Image.open(image_path)
    W, H = img.size
    stem = Path(image_path).stem
    out_path = Path(image_path).parent / f"{stem}_corrected.txt"
    lines = []
    for _, row in df.iterrows():
        cls = row['class']
        cls_idx = CLASS_NAMES.index(cls) if cls in CLASS_NAMES else 0
        xc = ((row['x1'] + row['x2']) / 2) / W
        yc = ((row['y1'] + row['y2']) / 2) / H
        bw = (row['x2'] - row['x1']) / W
        bh = (row['y2'] - row['y1']) / H
        lines.append(f"{cls_idx} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    out_path.write_text("\n".join(lines))
    return f"Saved {len(lines)} boxes → {out_path.name}"

In [15]:
with gr.Blocks(title="CFU Classifier") as demo:
    gr.Markdown("# CFU Classifier\nUpload an image and click **Run inference**. Edit the table to correct detections — the preview updates automatically.")

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="filepath", label="Upload plate image")
            run_btn     = gr.Button("Run inference", variant="primary")
            counts_out  = gr.JSON(label="Counts")
            save_btn    = gr.Button("Save corrected annotations")
            save_status = gr.Textbox(label="Save status", interactive=False)

        with gr.Column(scale=2):
            preview = gr.Plot(label="Preview")
            box_table = gr.Dataframe(
                headers=['class', 'x1', 'y1', 'x2', 'y2'],
                datatype=['str', 'number', 'number', 'number', 'number'],
                col_count=(5, 'fixed'),
                label="Detections — edit cells, add rows, or delete rows to correct",
                interactive=True,
            )

    run_btn.click(
        fn=run_inference,
        inputs=image_input,
        outputs=[preview, box_table, counts_out]
    )
    box_table.change(
        fn=update_preview,
        inputs=[image_input, box_table],
        outputs=[preview, counts_out]
    )
    save_btn.click(
        fn=save_annotations,
        inputs=[image_input, box_table],
        outputs=save_status
    )

demo.launch()

/Users/mzhai72/Desktop/SC-RNA Seq/CFU-classifier/.venv/lib/python3.12/site-packages/gradio/components/dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
